In [1]:
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from models import get_model, get_embeddings
from pdf_chunk import text_spliter

e:\Rohanta_AI_workbook\adavance_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = get_model()                   # your Groq LLM
embedding_model = get_embeddings() 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2197.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
vectorstore = Chroma.from_documents(
    text_spliter,
    embedding_model,
    collection_name="hyde_store"
)

In [4]:
# Step 1: prompt that makes LLM generate a hypothetical answer
hyde_prompt = ChatPromptTemplate.from_template("""
You are an expert assistant. Write a short factual passage that
directly answers the following question. Write it as if it were
extracted from a real document. Do NOT say "I don't know" —
write your best guess as a confident statement.

Question: {question}

Passage:
""")

In [5]:
# Step 2: chain that generates the hypothetical document
hypothetical_doc_chain = (
    hyde_prompt
    | llm
    | StrOutputParser()
)

In [6]:
def hyde_retriever(query: str, k: int = 4):
    """
    1. Generate hypothetical answer from query
    2. Embed hypothetical answer
    3. Search vector store with that embedding
    4. Return real documents
    """
    # Generate hypothetical document
    hypothetical_doc = hypothetical_doc_chain.invoke({"question": query})
    print(f"\n--- Hypothetical doc ---\n{hypothetical_doc}\n")

    # Embed it and search
    real_docs = vectorstore.similarity_search(
        hypothetical_doc,   # ← search with the hypothetical text
        k=k
    )
    return real_docs

In [9]:
query = "what is Rohanta's current university?"
docs = hyde_retriever(query, k=5)


--- Hypothetical doc ---
**Extract from: University Affiliations Database (UAD)**

**Entry: Rohanta**

Rohanta is a notable individual with a strong academic background. As per our records, Rohanta is currently affiliated with the University of Dhaka, where they are pursuing a higher degree in their field of expertise. The University of Dhaka, located in Dhaka, Bangladesh, is one of the oldest and most prestigious institutions in the country, known for its academic excellence and research opportunities.

**Source:** University Affiliations Database (UAD), Updated: March 2023



In [10]:
docs

[Document(id='27a22243-613f-45a9-bd75-4c6f679cf787', metadata={'source': 'my_data.txt'}, page_content='Before transitioning fully into industry, Rohanta taught undergraduate Computer Science to 100+ students. He designed curriculum and lab exercises focused on Python, Data Structures, and algorithms. He conducted workshops on data science tools including NumPy, Pandas, and Matplotlib. He supervised applied AI projects involving NLP and data-driven solutions. He guided NLP-based student research projects. This phase gave him strong communication and mentoring skills that most engineers do not'),
 Document(id='00007452-3f1b-4fad-a50f-7ff3b258f056', metadata={'source': 'my_data.txt'}, page_content='Role: Assistant Professor\nInstitution: Guru Gobind Singh Foundation\nDuration: May 2019 to August 2022\nLocation: Nashik, India'),
 Document(id='e36b152a-1840-496a-95a0-0ccb65fb63c4', metadata={'source': 'my_data.txt'}, page_content='IDENTITY AND CONTACT\n\nFull Name: Rohanta Dinkar Bhamare\nT

In [11]:
# CELL 5 — Full LCEL chain with HyDE
# ════════════════════════════════════════════════════════════════
answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context: {context}
Question: {question}
""")

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

hyde_chain = (
    {
        "context": RunnableLambda(
            lambda q: format_docs(hyde_retriever(q, k=4))
        ),
        "question": RunnablePassthrough()
    }
    | answer_prompt
    | llm
    | StrOutputParser()
)

print(hyde_chain.invoke("what is Rohanta's current university?"))


--- Hypothetical doc ---
**Extract from University Directory Update (2023)**

Rohanta is a current student at the University of California, Berkeley, where he is pursuing a Bachelor of Science degree in Computer Science.

I don't know.


In [24]:
from langchain_classic.chains import HypotheticalDocumentEmbedder

In [25]:
#  This wraps the whole generate → embed pipeline into a single embedder
hyde_embedder = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embedding_model,
    prompt_key="web_search",   # built-in template, or pass custom prompt
)


In [26]:
# Use it exactly like a normal embedder — drop-in replacement
hyde_vectorstore = Chroma.from_documents(
    text_spliter,
    hyde_embedder,           # real docs still use normal embeddings
    collection_name="hyde_builtin"
)

In [27]:
# At query time, hyde_embedder auto-generates + embeds the hypothetical doc
hyde_retriever_builtin = hyde_vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [28]:
docs = hyde_retriever_builtin.invoke(query)

In [29]:
docs

[Document(id='5b4665fc-44f0-4b59-9c4e-cfd3c217059b', metadata={'source': 'my_data.txt'}, page_content='IDENTITY AND CONTACT\n\nFull Name: Rohanta Dinkar Bhamare\nTitle: AI Engineer\nLocation: Frankfurt, Germany\nPhone: +49 15566030379\nEmail: rohantabhamare22@gmail.com\nLinkedIn: www.linkedin.com/in/rohanta-bhamare-380611346\nGitHub: https://github.com/rohantabhamar\n\n\nPROFESSIONAL SUMMARY'),
 Document(id='f2fd5ddf-a40a-4082-978b-45a8d640432a', metadata={'source': 'my_data.txt'}, page_content='IDENTITY AND CONTACT\n\nFull Name: Rohanta Dinkar Bhamare\nTitle: AI Engineer\nLocation: Frankfurt, Germany\nPhone: +49 15566030379\nEmail: rohantabhamare22@gmail.com\nLinkedIn: www.linkedin.com/in/rohanta-bhamare-380611346\nGitHub: https://github.com/rohantabhamar\n\n\nPROFESSIONAL SUMMARY'),
 Document(id='0008cfa1-5d02-4874-abbd-a6a66deb1e61', metadata={'source': 'my_data.txt'}, page_content='IDENTITY AND CONTACT\n\nFull Name: Rohanta Dinkar Bhamare\nTitle: AI Engineer\nLocation: Frankfurt, G